# Building a Prompt Chain — IT Maintenance Notice

Prompt chaining passes the output of one model request into another. In this experiment, Stage 1 extracts facts from an IT maintenance note; Stage 2 turns those facts into an employee notice. Python makes the connection explicitly using `facts`. The same `get_completion(system_prompt, user_prompt)` helper used earlier is reused unchanged.

You will inspect the source note, predict what must be preserved, run each stage and compare its result with the original. You can then change only the writing instruction or remove a source detail. All inputs are embedded below; no CSV or earlier notebook state is needed. Offline mode returns labelled supplied examples. Optional local OpenAI mode generates fresh responses. Run from the top in a fresh kernel; do not treat supplied examples as measurements of model performance.

## 1. Prepare the input

Read the maintenance note before executing the chain. **Predict:** Which facts must reach the final notice? What should happen if Stage 1 omits the available alternative? Write your prediction here before running Stage 1.

In [ ]:
maintenance_note = """
Office Wi-Fi will be unavailable on Tuesday
from 09:00 to 10:00 while technicians replace a router.

Wired connections will remain available.
Employees who need help should contact the IT service desk.
"""


## 2. Reuse the existing helper

This cell copies the shared helper from Section 3.2 exactly, keeping the two-argument interface introduced in Section 2.3. It supplies offline sample lookup, optional local `.env` configuration and request records. You do not need to rewrite it. An edited prompt with no saved offline example returns a clear message, not an invented model response.

In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")


### Define the two stages

Stage 1 receives the original note. Stage 2 will receive only the first response. These strings are the same instructions as the book; they do not make any requests until passed to `get_completion()`.

In [ ]:
extraction_instructions = """
Extract these facts from the maintenance note:
- Affected service
- Day and time
- Reason for the interruption
- Available alternative
- Support contact

Use a short labelled list.
If a detail is not provided, write "Not provided".
Do not invent information.
"""

notice_instructions = """
Write a short IT maintenance notice for employees
using only the supplied facts.

Include a subject line and a clear, friendly message.
Explain the interruption, available alternative
and support contact.

Do not add a sender name, department name or sign-off unless explicitly supplied.
Do not invent missing details.
Do not describe a detail marked "Not provided"
as if it were known.
"""


### Load supplied examples

The next cell registers five author-written responses for the unchanged baseline and optional tasks. These are fixed teaching fixtures, not newly generated answers. Inspect the outputs after making your prediction. Edited prompts without a matching fixture require live mode or your own clearly labelled written response.

In [ ]:
# Author-written offline fixtures. Registering them makes no API call.
supplied_facts = """
Affected service: Office Wi-Fi
Day and time: Tuesday, 09:00 to 10:00
Reason for the interruption: Technicians are replacing a router
Available alternative: Wired connections remain available
Support contact: IT service desk
"""

supplied_notice = """
Subject: Office Wi-Fi maintenance on Tuesday

Office Wi-Fi will be unavailable on Tuesday from 09:00 to 10:00 while technicians replace a router. Wired connections will remain available during this time. If you need help, please contact the IT service desk.
"""

bullet_notice_instructions = """
Write a short IT maintenance notice for employees
using only the supplied facts.

Include a subject line followed by three short bullet points.
Explain the interruption, available alternative
and support contact.

Do not add a sender name, department name or sign-off unless explicitly supplied.
Do not invent missing details.
Do not describe a detail marked "Not provided"
as if it were known.
"""

supplied_bullet_notice = """
Subject: Office Wi-Fi maintenance on Tuesday

- Office Wi-Fi will be unavailable on Tuesday from 09:00 to 10:00 for a router replacement.
- Wired connections will remain available.
- For help, contact the IT service desk.
"""

note_without_contact = """
Office Wi-Fi will be unavailable on Tuesday
from 09:00 to 10:00 while technicians replace a router.

Wired connections will remain available.
"""

supplied_missing_facts = """
Affected service: Office Wi-Fi
Day and time: Tuesday, 09:00 to 10:00
Reason for the interruption: Technicians are replacing a router
Available alternative: Wired connections remain available
Support contact: Not provided
"""

supplied_missing_notice = """
Subject: Office Wi-Fi maintenance on Tuesday

Office Wi-Fi will be unavailable on Tuesday from 09:00 to 10:00 while technicians replace a router. Wired connections will remain available. No support contact was provided in the supplied information.
"""

register_sample(extraction_instructions, maintenance_note, supplied_facts)
register_sample(notice_instructions, supplied_facts, supplied_notice)
register_sample(bullet_notice_instructions, supplied_facts, supplied_bullet_notice)
register_sample(extraction_instructions, note_without_contact, supplied_missing_facts)
register_sample(notice_instructions, supplied_missing_facts, supplied_missing_notice)


### Choose offline or OpenAI mode

Keep `USE_OPENAI=False` in browser Python. To generate fresh responses locally, install the companion-root `requirements.txt`, copy `.env.example` to `.env` beside `COMPANION_VERSION`, and set your own `OPENAI_API_KEY` and `OPENAI_MODEL`. Then set `USE_OPENAI=True` below and run from the beginning. Standalone copies use `.env` beside the notebook. Never put a key into a cell or output.

The baseline makes **two requests**. `RUN_FORMAT_CHANGE=True` adds **one** request, reusing `facts`. `RUN_MISSING_CONTACT=True` adds **two** requests using the changed source note. Leave both optional switches off until you inspect the baseline. A full run with both enabled makes five requests, each capped at 1,000 output tokens; retries are disabled. These are independent optional exercises, not an automatic feedback loop. Repeated live execution incurs new usage.

In [ ]:
USE_OPENAI = False
RUN_FORMAT_CHANGE = False
RUN_MISSING_CONTACT = False

# No client or private configuration is loaded in offline mode.
if USE_OPENAI:
    if MODE != "live":
        configure_live()
else:
    if MODE == "live":
        clear_live()
    print("Offline mode: supplied teaching responses, no OpenAI requests.")


## 3. Stage 1 — extract facts

Run this cell first. Compare the result with the source note: service, Tuesday 09:00–10:00, router replacement, wired alternative and IT service desk. No calendar date, email address or phone number is supplied. The result is ordinary text, not yet structured data checked by Python.

In [ ]:
# Stage 1 sends the original note and stores its response.
facts = get_completion(extraction_instructions, maintenance_note)
print("INTERMEDIATE FACTS")
print(facts)


## 4. Stage 2 — write the notice

Read `facts` before continuing. The next call receives `facts`, not the original note or the earlier instructions. Check the output against both: did information disappear, or did unsupported detail appear? A correct-looking notice is not proof that extraction was complete.

In [ ]:
# Stage 2 explicitly receives the first response as its user input.
notice = get_completion(notice_instructions, facts)
print("EMPLOYEE NOTICE")
print(notice)


## 5. Follow the connection

The helper records requests without credentials. Inspect the first two records to see that the second input is the first response. This displays the messages, not hidden model reasoning. In live mode the helper labels each response; offline records are marked as supplied.

In [ ]:
# Display just the baseline messages so the data flow is easy to trace.
for number, record in enumerate(request_records[:2], start=1):
    print(f"\nSTAGE {number} — {record['mode']}")
    print("INPUT:")
    print(record["user_prompt"])
    print("OUTPUT:")
    print(record["response"])


### Check and explain

Check whether both outputs preserve Tuesday, 09:00–10:00, Wi-Fi unavailability, router replacement, available wired connections and the IT service desk. Look for invented contact details or a changed meaning, such as implying all network access is unavailable. If an error appears, identify the first stage where it occurs. A labelled list can be well formatted yet factually wrong.

Explain in your own words why sending `maintenance_note` instead of `facts` to Stage 2 would change this two-stage workflow.

## 6. Change only the presentation

**Predict first:** How should the notice change when a subject line and three short bullet points are requested? Which facts must stay the same?

Set `RUN_FORMAT_CHANGE=True` in the configuration cell and rerun that cell, then run the next cell. Do **not** rerun extraction: `facts` should remain unchanged. This uses one additional live request when enabled. The baseline variables remain available for comparison.

In [ ]:
if RUN_FORMAT_CHANGE:
    # Reuse the saved baseline facts; only the writing instruction changes.
    bullet_notice = get_completion(bullet_notice_instructions, facts)
    print("REVISED PRESENTATION")
    print(bullet_notice)
else:
    print("Presentation task not run. Inspect the baseline first.")


## 7. Remove the support contact

**Predict first:** What should each stage do when the source no longer names the IT service desk? Set `RUN_MISSING_CONTACT=True` in the configuration cell, rerun that cell and then run the next cell. The changed note is supplied as `note_without_contact`; the extraction and writing instructions remain unchanged. This starts a separate two-call chain without overwriting the baseline `facts` or `notice`.

Check for `Not provided` at the extraction stage. The notice must not invent an email address, telephone number or named employee. Inspect fresh live results rather than assuming the model obeyed the instruction.

In [ ]:
if RUN_MISSING_CONTACT:
    # A changed source requires extraction again before writing.
    missing_facts = get_completion(extraction_instructions, note_without_contact)
    print("FACTS WITHOUT CONTACT")
    print(missing_facts)
    missing_notice = get_completion(notice_instructions, missing_facts)
    print("NOTICE WITHOUT CONTACT")
    print(missing_notice)
else:
    print("Missing-information task not run.")


## 8. Save your observations

Write a short comparison: one fact preserved, one possible error, the first stage responsible, and the effect of your presentation change. Distinguish supplied text, live responses and your own writing. No automatic output checks or gate is introduced here; Section 6.3 develops that next.

Save or download your notebook. Run the last cell to close the live client after your work. To return to offline mode, set `USE_OPENAI=False` and rerun from the top.

In [ ]:
# Close the client without making any request or erasing displayed outputs.
clear_live()
